# Fuel Lattice Parameter — Data Featurization

Builds the featurized Materials Project dataset used to train the lattice-parameter
models: queries MP for all non-deprecated, non-noble-gas compounds, extracts symmetry
and lattice parameters via `SpacegroupAnalyzer`, one-hot-encodes crystal system and
space group, and featurizes each composition with matminer.

All of the actual logic lives in `engine/build_dataset.py` — this notebook is a thin,
narrated wrapper around it. There is no hyperparameter or file path defined here that
isn't also defined in `engine/`; if you need to change the build behavior, change it
there so `cli.py build` and this notebook stay identical.

**A fresh clone of this repository does not need to run this notebook at all** — the
post-query resume point (`Data/MP_Dataset_Original_Trimmed.csv`) ships pre-committed,
so `python cli.py build --resume` only re-runs the ~9-minute featurization stage. This
notebook is useful if you want to inspect the intermediate dataframes interactively, or
force a full re-query of Materials Project (`--force`).

In [ ]:
import sys
from pathlib import Path

# Make `engine` importable when this notebook's kernel cwd is the repo root
sys.path.insert(0, str(Path.cwd()))

from engine import config, build_dataset
import pandas as pd

## Run the build

`build_dataset.build()` is exactly what `cli.py build` calls. `resume=True` (the
default) skips the Materials Project query entirely if the trimmed dataset already
exists — either the committed seed in `Data/`, or a previous run's output in
`Dataset/`. Pass `force=True` to ignore that and re-query Materials Project (requires
`MP_API_KEY` in `.env`).

In [ ]:
result = build_dataset.build(resume=True, force=False)
print(f"Rows in featurized dataset: {result['rows']}")
for k, v in result['paths'].items():
    print(f"  {k}: {v}")

## Inspect the result

In [ ]:
df = pd.read_csv(config.DATASET_FEATURIZED, low_memory=False)
display(df.head())
display(df.groupby('nelements').count().iloc[:, :1])

In [ ]:
import joblib
feature_labels = joblib.load(config.FEATURELABELS)
print(f"{len(feature_labels)} feature labels")
print(feature_labels[:20], '...')